# PI Controllers (PI_1 and PI_2)

Cheat sheet for some of the parameters symbols
α  β  ρ  θ  ξ  δ
₀ ₁ ₂ ₃ ₄ ₅ ₆ ₇ ₈ ₉

**Transfer function (PI_1, crossed):**
G_c(s) = [α₁(s+1) + β·α₂] / [(s+1)² − β·k]

**Key gains:**
- K_P_eff = α₁ + β·α₂  (effective proportional numerator)
- K_I_eff = α₁          (integral-like coefficient)

**Steady-state error:**
e_ss = r / (1 + θ₁θ₂·(α₁+β·α₂)/(1−β·k))

**Zero SS error requires β·k = 1** (not automatic — depends on parameter choice).
At default β=0.3, k=0.3: β·k = 0.09, so there IS steady-state error.

**PI_1 (crossed):** q₁ driven by y and u₁.
**PI_2 (uncrossed):** q₂ driven by ref and u₂.

**Stability boundary (Routh–Hurwitz, 4th-order + integrator pole):** α₁·θ₁·θ₂ < **6**.

## Setup

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import numpy as np
import warnings
import sympy as sp
from scipy.optimize import differential_evolution, NonlinearConstraint
warnings.filterwarnings('ignore')
from bokeh.io import output_notebook, show

import biomolecular_controllers as bc
    
print('✓ Imports successful!')
from biomolecular_controllers.figure_saver import save_fig
from biomolecular_controllers.figure_gifs import save_gif


output_notebook()



✓ Imports successful!


Loading BokehJS ...

In [3]:
output_notebook()

runner  = bc.SimulationRunner()
calc    = bc.MetricsCalculator()
stab    = bc.StabilityAnalyzer()
plotter = bc.VisualizationPipeline()
swiffer = bc.SensitivityAnalyzer()

print('✓ Tools initialized!')

Loading BokehJS ...

✓ Tools initialized!


## Parameter sweep

In [4]:
# Set fixed params such that contribution to any gain(s) not being investigated is ~0.1 or less
# Effective Gain K_p = alpha_1 - alpha_2*beta  
# Effectiev Gain K_i = 1 / (beta*k - 1)
# Stability: 8*(-beta*alpha_2/alpha_1 + sqrt((beta*alpha_2/alpha_1)^2 + 1)) > alpha_1*theta_1*theta_2

alpha_1, alpha_2, beta, k, theta_1, theta_2, xi = sp.symbols(
    'alpha_1 alpha_2 beta k theta_1 theta_2 xi', positive=True
)

q  = beta * k
K1 = alpha_1 * theta_1 * theta_2
K2 = alpha_2 * beta * theta_1 * theta_2


rh_conditions = {
    'P_LPF': 8 - K1,   # reduces to P+LPF bound when alpha_2=beta=k=0
    'PI': 1 - q + K1 - K2  # q=1 case
}

base_params = {
    theta_1: 1.0,
    theta_2: 1.0,   # theta_1*theta_2 = 1, gain reads directly
    beta:   1.0,   # beta=1 simplifies beta*k = k, beta*alpha_2 = alpha_2
    k:      1.001, # beta*k = 1 almost exactly, integral action active
    alpha_2: 0.1,   # small but nonzero, contributes 0.1 within K_p subtraction
    alpha_1: 1.0,   # initialization, overridden when sweeping
}


# example with simulation parameters
# sweep alpha_1, fix everything else
lower, upper = swiffer.get_feasible_range(rh_conditions, alpha_1, base_params)

inside, out_low, out_high, combined = swiffer.build_sweep(lower, upper, n_inside=5)
print(f"Feasible range for alpha_1 with small excursion above/below stable regime: ({lower:.4f}, {upper:.4f})")
print(f"Inside:  {np.round(inside, 3)}")
print(f"Below:   {np.round(out_low, 3)}")
print(f"Above:   {np.round(out_high, 3)}")


  P_LPF: upper bound at 8.0000
  PI: lower bound at 0.1010
Feasible range for alpha_1 with small excursion above/below stable regime: (0.1010, 8.0000)
Inside:  [0.496 2.273 4.051 5.828 7.605]
Below:   [-1.084]
Above:   [9.185]


## Optimal Parameter Set

In [36]:
alpha_1, alpha_2, beta, k, theta_1, theta_2 = sp.symbols(
    'alpha_1 alpha_2 beta k theta_1 theta_2', positive=True
)

# ss error in fast sequestration limit: e_ss = r / (1 - theta_1*theta_2*(alpha_1-alpha_2*beta)/(beta*k-1))
r_val = 1.0
F = alpha_1 - alpha_2*beta
G = beta*k - 1
ss_error_PI = r_val / (1 - (theta_1*theta_2*F)/G)

K1 = alpha_1*theta_1*theta_2
K2 = alpha_2*beta*theta_1*theta_2
q  = beta*k

rh_conditions_PI = {
    'P_LPF': 8 - K1,
    'a0':    1 - q + K1 - K2
}

fixed = {theta_1: 1.0, theta_2: 1.0}
param_syms = [alpha_1, alpha_2, beta, k]

objective, constraints, bounds = swiffer.build_optimizer(
    'PI_1', fixed, rh_conditions_PI, ss_error_PI, param_syms
)

# use differential evolution for global search within bounds

result_global = differential_evolution(
    objective,
    bounds=bounds,
    constraints=constraints,
    maxiter=1000,
    tol=1e-8,
    polish=True   # refines with L-BFGS-B after global search
)

print(f"Optimal parameters:")
for sym, val in zip(param_syms, result_global.x):
    print(f"  {sym} = {val:.4f}")
print(f"Minimum SS error: {result_global.fun:.6f}")
print(f"Optimization success: {result_global.success}")

# verify stability at optimum
print("\nRH conditions at optimum (all should be > 0):")
for name, cond in rh_conditions_PI.items():
    val = float(cond.subs(fixed).subs(
        dict(zip(param_syms, result_global.x))
    ))
    print(f"  {name}: {val:.4f}")

Optimal parameters:
  alpha_1 = 3.7153
  alpha_2 = 1.4759
  beta = 1.2147
  k = 2.4059
Minimum SS error: -4503599627370493.500000
Optimization success: True

RH conditions at optimum (all should be > 0):
  P_LPF: 4.2847
  a0: 0.0000


## Part 1: Sweep α₁ (Vary Gain) — PI_1

G = α₁·θ₁·θ₂.  Stability requires G < 6.  Unlike PC, steady-state error → 0 for all stable configs.
The tradeoff shifts to **overshoot vs settling time** as α₁ increases toward the boundary.

In [5]:
# Sweep alpha_1 
alpha_vals = inside[::-1]

# Step input
ref_low  = bc.DEFAULT_PARAMS['PI_1']['ref']
ref_high = 50.0
t_step   = 150.0
t_span   = (0, 600)

print(f'Sweeping α₁: {max(alpha_vals):.2f} → {min(alpha_vals):.2f}')
print(alpha_vals)

Sweeping α₁: 7.61 → 0.50
[7.60505  5.827775 4.0505   2.273225 0.49595 ]


In [6]:
trajectories_dict   = {}
gains_list          = []
ss_errors_list      = []
ss_stds_list  = []
settling_times_list = []
overshoots_list     = []
rise_times_list     = []

params = {
    'theta_1': 1.0,
    'theta_2': 1.0,
    'beta':    1.0,
    'k':       1.0 + 1e-2,
    'alpha_2': 0.1563,
    'alpha_1': 1.0,
}
print('Running PI_1 simulations...')

for model_key in ['PI_1', 'PI_2']:
    for alpha in alpha_vals:
        current_params = {**params, 'alpha_1': alpha}
        perturbations = [
            {'time': t_step, 'type': 'parameter', 'param': 'ref', 'value': ref_high}
        ]
        try:
            result = runner.run_with_perturbations(
                model=model_key, t_span=t_span, points=1200,
                perturbations=perturbations, params=current_params
            )
            time = np.asarray(result['time'],        dtype=float)
            y    = np.asarray(result['states']['y'], dtype=float)
            ref  = np.where(time < t_step, ref_low, ref_high)

            # Effective gain numerator: K_P_eff·θ₁θ₂ = (α₁+β·α₂)·θ₁θ₂
            # Full DC gain for SS error: K_P_eff·θ₁θ₂ / (1−β·k)
            K_P_eff_num = bc.gain.gain_pi1(current_params)

            # Use actual SS value for rise_time (SS error may be non-zero when β·k ≠ 1)
            y_ss = float(np.mean(y[-20:]))

            os_r = calc.overshoot(time, y, ref=ref_high, pert_start=t_step, pert_end=t_span[1])
            st_r = calc.settling_time(time, y, ref=ref_high, pert_start=t_step, pert_end=t_span[1])
            rt_r = calc.rise_time(time, y, ref=y_ss, pert_start=t_step, pert_end=t_span[1])
            ss_r = calc.steady_state(time,y, pert_start=t_step, pert_end=t_span[1])

            trajectories_dict[alpha] = {'time': time, 'y': y, 'ref': ref}
            gains_list.append(K_P_eff_num)
            ss_errors_list.append(abs(ss_r['ss_value'] - ref_high))
            ss_stds_list.append(ss_r['ss_std'])
            settling_times_list.append(st_r['settling_time'] if st_r['settled'] else np.nan)
            overshoots_list.append(os_r['magnitude'])
            rise_times_list.append(rt_r['rise_time'] if rt_r['valid'] else np.nan)
            
        except Exception as e:
            import traceback
            traceback.print_exc()
            print(f'  α₁={alpha:.2f}: failed ({str(e)[:50]})')
            gains_list.append(bc.gain.gain_pi1(params))
            ss_errors_list.append(np.nan)
            ss_stds_list.append(np.nan)
            settling_times_list.append(np.nan)
            overshoots_list.append(np.nan)
            rise_times_list.append(np.nan)

gains          = np.array(gains_list)
ss_errors      = np.array(ss_errors_list)
ss_stds        = np.array(ss_stds_list)
settling_times = np.array(settling_times_list)
overshoots     = np.array(overshoots_list)
rise_times     = np.array(rise_times_list)

print(f'✓ Completed {np.sum(~np.isnan(ss_errors))}/{len(alpha_vals)} simulations')
print(f'SS errors: {ss_errors}')

Running PI_1 simulations...
✓ Completed 10/5 simulations
SS errors: [24.14779095  0.08637619  0.12871527  0.23730114  1.51674054 26.27947925
 18.04637229  0.11911965  0.2066421   0.77850288]


## Plot 1: Steady-State Error vs Gain

**e_ss = r / (1 + θ₁θ₂·(α₁+β·α₂)/(1−β·k))**

With β=0.3, k=0.3: β·k=0.09, so SS error is non-zero.
Zero SS error only occurs when β·k = 1.
Higher α₁ → larger K_P_eff → smaller SS error, but approaching instability at α₁·θ₁θ₂ = 6.

In [7]:
layout, traj_renderers, ref_renderers, metric_source = plotter.plot_metric_interactive( # type: ignore[misc]
    param_vals=np.array(alpha_vals),
    trajectories=trajectories_dict,
    metric_name='Steady State Error',
    gain_vals=gains,
    metric_vals=ss_errors,
    metric_stds=ss_stds,
    title='PI_1: Steady-State Error vs K_I  [e_ss = r/(1-θ₁θ₂(α₁-α₂β)/(1−βk))]',
    x_label='K_I = θ₁θ₂(α₁−βα₂) / (1−βk)',
    x_scale='linear',
    y_scale='linear',
    return_sources=True,
)

show(layout)
#save_fig(layout, 'PI_1', 'steady_state_error', fmt='png', show=True)
save_gif(layout, traj_renderers, ref_renderers, metric_source, 'PI_1', 'steady_state_error')


[0, 1, 2, 3, 4]


  Saved GIF  → C:\Users\natha\OneDrive\Documents\CollegeDocuments\Spring_2026_CMU\Samaniego Lab\biomolecular_controllers\figures\pi_1_steady_state_error.gif


WindowsPath('C:/Users/natha/OneDrive/Documents/CollegeDocuments/Spring_2026_CMU/Samaniego Lab/biomolecular_controllers/figures/pi_1_steady_state_error.gif')

## Plot 2: Settling Time vs Gain

In [40]:
layout, traj_renderers, ref_renderers, metric_source = plotter.plot_metric_interactive( # type: ignore[misc]
    param_vals=np.array(alpha_vals),
    trajectories=trajectories_dict,
    metric_name='Settling Time',
    gain_vals=gains,
    metric_vals=settling_times,
    title='PI_1: Settling Time vs K_I  [e_ss = r/(1-θ₁θ₂(α₁-α₂β)/(1−βk))]',
    x_label='K_I = θ₁θ₂(α₁−βα₂) / (1−βk)',
    x_scale='linear',
    y_scale='linear',
    return_sources=True,
)

show(layout)
#save_fig(layout, 'PI_1', 'settling_time', fmt='png', show=True)
save_gif(layout, traj_renderers, ref_renderers, metric_source, 'PI_1', 'settling_time')


[0, 1, 2, 3, 4]


  Saved GIF  → C:\Users\natha\OneDrive\Documents\CollegeDocuments\Spring_2026_CMU\Samaniego Lab\biomolecular_controllers\figures\pi_1_settling_time.gif


WindowsPath('C:/Users/natha/OneDrive/Documents/CollegeDocuments/Spring_2026_CMU/Samaniego Lab/biomolecular_controllers/figures/pi_1_settling_time.gif')

## Plot 3: Overshoot vs Gain

In [41]:
layout, traj_renderers, ref_renderers, metric_source = plotter.plot_metric_interactive( # type: ignore[misc]
    param_vals=np.array(alpha_vals),
    trajectories=trajectories_dict,
    metric_name='Overshoot',
    gain_vals=gains,
    metric_vals=overshoots,
    title='PI_1: Overshoot vs K_I  [e_ss = r/(1-θ₁θ₂(α₁-α₂β)/(1−βk))]',
    x_label='K_I = θ₁θ₂(α₁−βα₂) / (1−βk)',
    x_scale='linear',
    y_scale='linear',
    return_sources=True,
)

show(layout)
#save_fig(layout, 'PI_1', 'overshoot', fmt='png', show=True)
save_gif(layout, traj_renderers, ref_renderers, metric_source, 'PI_1', 'overshoot')


[0, 1, 2, 3, 4]


  Saved GIF  → C:\Users\natha\OneDrive\Documents\CollegeDocuments\Spring_2026_CMU\Samaniego Lab\biomolecular_controllers\figures\pi_1_overshoot.gif


WindowsPath('C:/Users/natha/OneDrive/Documents/CollegeDocuments/Spring_2026_CMU/Samaniego Lab/biomolecular_controllers/figures/pi_1_overshoot.gif')

## Plot 4: Rise Time vs Gain

In [42]:
layout, traj_renderers, ref_renderers, metric_source = plotter.plot_metric_interactive( # type: ignore[misc]
    param_vals=np.array(alpha_vals),
    trajectories=trajectories_dict,
    metric_name='Rise Time',
    gain_vals=gains,
    metric_vals=rise_times,
    title='PI_1: Rise Time vs K_I  [e_ss = r/(1-θ₁θ₂(α₁-α₂β)/(1−βk))]',
    x_label='K_I = θ₁θ₂(α₁−βα₂) / (1−βk)',
    x_scale='linear',
    y_scale='linear',
    return_sources=True,
)

show(layout)
#save_fig(layout, 'PI_1', 'rise_time', fmt='png', show=True)
save_gif(layout, traj_renderers, ref_renderers, metric_source, 'PI_1', 'rise_time')


[0, 1, 2, 3, 4]


  Saved GIF  → C:\Users\natha\OneDrive\Documents\CollegeDocuments\Spring_2026_CMU\Samaniego Lab\biomolecular_controllers\figures\pi_1_rise_time.gif


WindowsPath('C:/Users/natha/OneDrive/Documents/CollegeDocuments/Spring_2026_CMU/Samaniego Lab/biomolecular_controllers/figures/pi_1_rise_time.gif')

## Constrained Sweeps

In [43]:
# for debugging
# net effect: pure integral action with varying beta/k ratio
alpha_1, alpha_2, beta, k, theta_1, theta_2 = sp.symbols(
    'alpha_1 alpha_2 beta k theta_1 theta_2', positive=True
)

KP_expr = theta_1 * theta_2 * (alpha_1 - alpha_2*beta)
KI_expr = beta * k   # =1 for integral action

# sweep beta, solve for k and alpha_2
beta_vals = np.linspace(0.3, 2.0, 8)
fixed = {theta_1: 1.0, theta_2: 1.0, alpha_1: 2.0}

KP_target = 1.5
sweep_beta = swiffer.make_constrained_sweep(
    free_param_vals  = beta_vals,
    free_param_sym   = beta,
    gain_constraints = {
        'integral':     (KI_expr,  1.0),       # beta*k = 1
        'proportional': (KP_expr,  KP_target), # KP fixed
    },
    all_params_sym   = [alpha_2, beta, k],
    fixed_params     = fixed
)
# print what the sweep actually looks like
print(f"{'beta':>6} {'k':>6} {'alpha_2':>8} {'KP':>6} {'beta*k':>6}")
for p in sweep_beta:
    bk = p['beta'] * p['k']
    kp = (p['alpha_1'] - p['alpha_2']*p['beta']) * p['theta_1'] * p['theta_2']
    print(f"{p['beta']:>6.3f} {p['k']:>6.3f} {p['alpha_2']:>8.4f} "
          f"{kp:>6.3f} {bk:>6.3f}")

Symbolic solutions for dependent params:
  alpha_2 = 0.5⋅(2.0⋅α₁⋅θ₁⋅θ₂ - 3.0)
────────────────────────
        β⋅θ₁⋅θ₂         
  k = 1
─
β
  beta      k  alpha_2     KP beta*k
 0.300  3.333   1.6667  1.500  1.000
 0.543  1.842   0.9211  1.500  1.000
 0.786  1.273   0.6364  1.500  1.000
 1.029  0.972   0.4861  1.500  1.000
 1.271  0.787   0.3933  1.500  1.000
 1.514  0.660   0.3302  1.500  1.000
 1.757  0.569   0.2846  1.500  1.000
 2.000  0.500   0.2500  1.500  1.000


In [44]:
# Sweep alpha_1 
alpha_1, alpha_2, beta, k, theta_1, theta_2 = sp.symbols(
    'alpha_1 alpha_2 beta k theta_1 theta_2', positive=True
)

KP_expr = theta_1 * theta_2 * (alpha_1 - alpha_2*beta)
KI_expr = beta * k   # =1 for integral action

fixed = {theta_1: 1.0, theta_2: 1.0, alpha_1: 2.0}

# sweep beta, solve for k and alpha_2
beta_vals = np.linspace(0.3, 2.0, 8)
fixed = {theta_1: 1.0, theta_2: 1.0, alpha_1: 2.0}

KP_target = 1.5
sweep_beta = swiffer.make_constrained_sweep(
    free_param_vals  = beta_vals,
    free_param_sym   = beta,
    gain_constraints = {
        'integral':     (KI_expr,  1.0),       # beta*k = 1
        'proportional': (KP_expr,  KP_target), # KP fixed
    },
    all_params_sym   = [alpha_2, beta, k],
    fixed_params     = fixed
)

# Step input
ref_low  = bc.DEFAULT_PARAMS['PI_1']['ref']
ref_high = 50.0
t_step   = 150.0
t_span   = (0, 600)


trajectories_dict   = {}
gains_list          = []
ss_errors_list      = []
ss_stds_list  = []
settling_times_list = []
overshoots_list     = []
rise_times_list     = []

params = {
    'theta_1': 1.0,
    'theta_2': 1.0,
    'beta':    1.0,
    'k':       0.9,
    'alpha_2': 0.1563,
    'alpha_1': 1.0,
}
print('Running PI_1 simulations...')
print(sweep_beta)
for sweep_point in sweep_beta:
    current_params = {**sweep_point}
    beta_val = current_params['beta']   # extract the swept value for trajectory key
    perturbations = [
        {'time': t_step, 'type': 'parameter', 'param': 'ref', 'value': ref_high}
    ]
    try:
        result = runner.run_with_perturbations(
            model='PI_1', t_span=t_span, points=1200,
            perturbations=perturbations, params=current_params
        )
        time = np.asarray(result['time'],        dtype=float)
        y    = np.asarray(result['states']['y'], dtype=float)
        ref  = np.where(time < t_step, ref_low, ref_high)

        # Effective gain numerator: K_P_eff·θ₁θ₂ = (α₁+β·α₂)·θ₁θ₂
        # Full DC gain for SS error: K_P_eff·θ₁θ₂ / (1−β·k)
        K_P_eff_num = bc.gain.gain_pi1(current_params)

        # Use actual SS value for rise_time (SS error may be non-zero when β·k ≠ 1)
        y_ss = float(np.mean(y[-20:]))

        os_r = calc.overshoot(time, y, ref=ref_high, pert_start=t_step, pert_end=t_span[1])
        st_r = calc.settling_time(time, y, ref=ref_high, pert_start=t_step, pert_end=t_span[1])
        rt_r = calc.rise_time(time, y, ref=y_ss, pert_start=t_step, pert_end=t_span[1])
        ss_r = calc.steady_state(time,y, pert_start=t_step, pert_end=t_span[1])

        trajectories_dict[float(beta_val)] = {'time': time, 'y': y, 'ref': ref}
        gains_list.append(K_P_eff_num)
        ss_errors_list.append(abs(ss_r['ss_value'] - ref_high))
        ss_stds_list.append(ss_r['ss_std'])
        settling_times_list.append(st_r['settling_time'] if st_r['settled'] else np.nan)
        overshoots_list.append(os_r['magnitude'])
        rise_times_list.append(rt_r['rise_time'] if rt_r['valid'] else np.nan)
    except Exception as e:
        gains_list.append(bc.gain.gain_pi1(params))
        ss_errors_list.append(np.nan)
        ss_stds_list.append(np.nan)
        settling_times_list.append(np.nan)
        overshoots_list.append(np.nan)
        rise_times_list.append(np.nan)

gains          = np.array(gains_list)
ss_errors      = np.array(ss_errors_list)
ss_stds        = np.array(ss_stds_list)
settling_times = np.array(settling_times_list)
overshoots     = np.array(overshoots_list)
rise_times     = np.array(rise_times_list)

beta_vals = [float(p['beta']) for p in sweep_beta]
beta_vals_arr = np.array([float(p['beta']) for p in sweep_beta])

layout, traj_renderers, ref_renderers, metric_source = plotter.plot_metric_interactive( # type: ignore[misc]
    param_vals=np.array(beta_vals),
    trajectories=trajectories_dict,
    param_name='β',
    metric_name='Steady State Error',
    gain_vals=beta_vals_arr,
    metric_vals=ss_errors,
    metric_stds=ss_stds,
    title='PI_1: Steady-State Error vs K_I  [e_ss = r/(1-θ₁θ₂(α₁-α₂β)/(1−βk))]',
    x_label='K_I = θ₁θ₂(α₁−βα₂) / (1−βk)',
    x_scale='linear',
    y_scale='linear',
)

save_fig(layout, 'PI_1', 'constrained_sweep_beta', fmt='png', show=True)


Symbolic solutions for dependent params:
  alpha_2 = 0.5⋅(2.0⋅α₁⋅θ₁⋅θ₂ - 3.0)
────────────────────────
        β⋅θ₁⋅θ₂         
  k = 1
─
β
Running PI_1 simulations...
[{'theta_1': 1.0, 'theta_2': 1.0, 'alpha_1': 2.0, 'beta': 0.3, 'alpha_2': 1.6666666666666667, 'k': 3.3333333333333335}, {'theta_1': 1.0, 'theta_2': 1.0, 'alpha_1': 2.0, 'beta': 0.5428571428571428, 'alpha_2': 0.9210526315789475, 'k': 1.842105263157895}, {'theta_1': 1.0, 'theta_2': 1.0, 'alpha_1': 2.0, 'beta': 0.7857142857142857, 'alpha_2': 0.6363636363636364, 'k': 1.2727272727272727}, {'theta_1': 1.0, 'theta_2': 1.0, 'alpha_1': 2.0, 'beta': 1.0285714285714285, 'alpha_2': 0.48611111111111116, 'k': 0.9722222222222223}, {'theta_1': 1.0, 'theta_2': 1.0, 'alpha_1': 2.0, 'beta': 1.2714285714285714, 'alpha_2': 0.39325842696629215, 'k': 0.7865168539325843}, {'theta_1': 1.0, 'theta_2': 1.0, 'alpha_1': 2.0, 'beta': 1.5142857142857142, 'alpha_2': 0.33018867924528306, 'k': 0.6603773584905661}, {'theta_1': 1.0, 'theta_2': 1.0, 'alpha_

TypeError: cannot unpack non-iterable Column object

## Part 2: Stability Diagram (α₁ vs θ₁)


In [ ]:
# PI_1 Parameter stability diagram
alpha_range = np.linspace(0.1, 10, 400)
theta_range = np.linspace(0.1, 10, 400)
theta_2_fixed = 1.0

upper_fn, lower_fn = bc.gain.pi1_stability_boundary(
    theta_2=theta_2_fixed, alpha_2=0.1, beta=1.0, k=0.999
)

p = plotter.plot_stability_diagram(
    x_vals=alpha_range,
    y_vals=theta_range,
    stable_condition=lambda A, T: (
        (A * T * theta_2_fixed < 8) &           # P_LPF
        ((A - 0.1) * T * theta_2_fixed > -0.001) # a0, beta*k~1
    ),
    boundary_fns=[
        {
            'fn':    upper_fn,
            'label': 'PI upper boundary',
            'color': 'black',
            'dash':  'dashed',
            'line_width': 2.0,
        },
        {
            'fn':    lower_fn,
            'label': 'PI lower boundary  (a0)',
            'color': 'red',
            'dash':  'dashed',
            'line_width': 2.0,
        },
    ],
    x_name='α₁',
    y_name='θ₁',
    title=f'PI_1 Parameter Stability Diagram  (θ₂={theta_2_fixed})',
)
p.scatter([-999], [-999], marker="square", size=14,
          fill_color="#8ECF8E", fill_alpha=0.65,
          line_color=None, legend_label="Stable")
p.scatter([-999], [-999], marker="square", size=14,
          fill_color="#C8C8C8", fill_alpha=0.65,
          line_color=None, legend_label="Unstable")

save_fig(p, 'PI_1', 'stability_diagram', fmt='png', show=True)


  Saved PNG  → C:\Users\natha\OneDrive\Documents\CollegeDocuments\Spring_2026_CMU\Samaniego Lab\biomolecular_controllers\figures\pi_1_stability_diagram.png


WindowsPath('C:/Users/natha/OneDrive/Documents/CollegeDocuments/Spring_2026_CMU/Samaniego Lab/biomolecular_controllers/figures/pi_1_stability_diagram.png')

## Part 3: Root Locus (Dominant Eigenvalue)

The integrator adds a pole at the origin — watch how eigenvalues approach the imaginary axis faster than PC as α₁ increases.

In [ ]:
root_alpha_vals = np.linspace(0.1, 5.5, 12)

eigenvalue_paths = {}
for model_key in ['PI_1', 'PI_2']:
    reals, imags, gs = [], [], []
    for val in root_alpha_vals:
        p_params = {
            'alpha_1': val,
            'alpha_2': 2.0,
            'theta_1': 1.0,
            'theta_2': 1.0,
            'beta':    1.0,
            'k':       1.001,
        }
        try:
            res  = stab.analyze_stability(model=model_key, params=p_params, steady_state_time=100.0)
            eigs = res.eigenvalues
            dom  = eigs[np.argmax(np.real(eigs))]
            reals.append(float(np.real(dom)))
            imags.append(float(np.imag(dom)))
            gs.append(res.gain)
        except Exception:
            reals.append(np.nan); imags.append(np.nan); gs.append(np.nan)
    eigenvalue_paths[model_key] = {
        'real': np.array(reals),
        'imag': np.array(imags),
        'gain': np.array(gs),
    }

fig = plotter.plot_root_locus(
    eigenvalue_paths,
    title='PI Controllers Root Locus: Dominant Eigenvalue as α₁ varies'
)
show(fig)
save_fig(fig, 'PI_controllers', 'root_locus', fmt='png', show=True)


  Saved PNG  → C:\Users\natha\OneDrive\Documents\CollegeDocuments\Spring_2026_CMU\Samaniego Lab\biomolecular_controllers\figures\pi_controllers_root_locus.png


WindowsPath('C:/Users/natha/OneDrive/Documents/CollegeDocuments/Spring_2026_CMU/Samaniego Lab/biomolecular_controllers/figures/pi_controllers_root_locus.png')